#Ollama:
Ollama lets you run LLMs locally on your machine (Mac/Linux/Windows) in a CLI-style chat interface.

🔁 Exported the Hugging Face fine-tuned model and convert it to GGUF or Ollama-compatible format using transformers + transformers-to-ollama.

In [ ]:
# Install dependencies
!pip install -q transformers datasets

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Load Model

In [ ]:
# ✅ Load model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Load Dataset

In [ ]:
# ✅ Create synthetic mental health support dataset
data = {
    "text": [
        "User: I'm feeling really anxious lately.\nAssistant: I'm really sorry you're feeling this way. You're not alone, and I'm here for you.",
        "User: I don't know how to deal with stress.\nAssistant: It's okay to feel overwhelmed. Taking small steps like deep breathing can help.",
        "User: I feel like nobody understands me.\nAssistant: That must be really hard. But please know your feelings are valid and someone does care.",
        "User: I can't sleep because of my worries.\nAssistant: Sleep can be tough when the mind is racing. Sometimes journaling or calming music can help.",
    ]
}

In [ ]:
# Wrap in Hugging Face Dataset
dataset = Dataset.from_dict(data)

# Tokenize Dataset

In [ ]:
# Tokenize
def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_fn)

# Setup Training

In [ ]:
# ✅ Training setup
training_args = TrainingArguments(
    output_dir="./mental-health-chatbot",
    per_device_train_batch_size=1,
    num_train_epochs=2,
    logging_steps=1,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# Model Finetuning

In [ ]:
# Fine-tune the model
trainer.train()

# Inference - Chat With the Bot

In [ ]:
prompt = "User: I'm feeling very stressed at work.\nAssistant:"
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

# Set pad_token_id explicitly and add attention_mask
output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,         # Enable sampling to avoid repetition
    top_k=50,                # Top-k sampling
    top_p=0.95,              # Nucleus sampling
    temperature=0.7          # Lower temp = less randomness
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


Simulated fine-tuning a mental health support chatbot, teaching a model to respond empathetically using distilgpt2.

✅ Used prompt + high-quality assistant responses to teach supportive behavior.

✅ Adapted to Hugging Face + Transformers.

- References: https://medium.com/@mauryaanoop3/fine-tuning-microsoft-phi3-with-unsloth-for-mental-health-chatbot-development-ddea4e0c46e7

# Export to Ollama

In [ ]:
import os

checkpoints = [ckpt for ckpt in os.listdir("./mental-health-chatbot") if ckpt.startswith("checkpoint")]
print(checkpoints)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./mental-health-chatbot/checkpoint-8"

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)


In [ ]:
model.save_pretrained("./mental-health-model")
tokenizer.save_pretrained("./mental-health-model")


In [ ]:
!zip -r mental_health_model.zip ./mental-health-model
from google.colab import files
files.download("mental_health_model.zip")


In [ ]:
pip install transformers transformers-to-ollama

unzip unsloth_phi2_export.zip

transformers-to-ollama \
  --model ./unsloth_phi2_export \
  --output ./phi2-ollama \
  --license mit \
  --format gguf

ollama create phi2-chatbot -f ./phi2-ollama/Ollamafile
ollama run phi2-chatbot
